# 1. 加载训练数据


In [ ]:
import os
import pathlib
import requests
import time
import tqdm
import numpy as np
import gzip
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
train_images_filename = "data/train-images-idx3-ubyte.gz"
train_labels_filename = "data/train-labels-idx1-ubyte.gz"
test_images_filename = "data/t10k-images-idx3-ubyte.gz"
test_labels_filename = "data/t10k-labels-idx1-ubyte.gz"

validation_size = 5000


def read_mnist_image_set(images_filename):
    with gzip.GzipFile(images_filename, "rb") as gz:
        magic = int.from_bytes(gz.read(4), "big")
        assert magic == 2051, "Not an MNIST image set"

        num_images = int.from_bytes(gz.read(4), "big")
        rows = int.from_bytes(gz.read(4), "big")
        cols = int.from_bytes(gz.read(4), "big")

        data = np.frombuffer(gz.read(num_images * rows * cols), dtype=np.uint8)
        data = np.reshape(data, (num_images, rows * cols))
        return data


def read_mnist_label_set(labels_filename):
    with gzip.GzipFile(labels_filename, "rb") as gz:
        magic = int.from_bytes(gz.read(4), "big")
        assert magic == 2049, "Not an MNIST label set"

        num_labels = int.from_bytes(gz.read(4), "big")
        labels = np.frombuffer(gz.read(num_labels), dtype=np.uint8)
        return labels


print("\nLoading train set ...")
train_data = read_mnist_image_set(train_images_filename) / np.float32(255)
train_labels = read_mnist_label_set(train_labels_filename)
train_data, val_data = train_data[:-validation_size], train_data[-validation_size:]
train_labels, val_labels = (
    train_labels[:-validation_size],
    train_labels[-validation_size:],
)
print(f"train_data:   [{str(train_data.dtype)}] {train_data.shape}")
print(f"train_labels: [{str(train_labels.dtype)}] {train_labels.shape}")
print(f"val_data:     [{str(val_data.dtype)}] {val_data.shape}")
print(f"val_labels:   [{str(val_labels.dtype)}] {val_labels.shape}")

print("\nLoading test set ...")
test_data = read_mnist_image_set(test_images_filename) / np.float32(255)
test_labels = read_mnist_label_set(test_labels_filename)
print(f"test_data:   [{str(test_data.dtype)}] {test_data.shape}")
print(f"test_labels: [{str(test_labels.dtype)}] {test_labels.shape}")


In [ ]:
# Preview dataset

_ = plt.figure(figsize=(6, 6))
_ = plt.title("MNIST Preview")
for label in range(10):
    for img_index, img_data in enumerate(train_data[train_labels == label][:10]):
        _ = plt.imshow(
            img_data.reshape(28, 28),
            "gray",
            vmin=0,
            vmax=1,
            interpolation="nearest",
            extent=(img_index, img_index + 1, label, label + 1),
        )
_ = plt.xticks([])
_ = plt.yticks(np.arange(10) + 0.5, range(10))
_ = plt.xlim(0, 10)
_ = plt.ylim(0, 10)

# 2. 实现网络架构


## 2.0 神经网络模型框架


In [ ]:
class Network:
    """
    简单的神经网络模型
    """

    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, x):
        # 逐层前向计算
        for i in range(len(self.layers)):
            x = self.layers[i].forward(x)
        return x

    def backward(self, delta):
        # 逐层后向计算
        for i in reversed(range(len(self.layers))):  # 逆向遍历
            delta = self.layers[i].backward(delta)

## 2.1 `SGD` 随机梯度下降


In [ ]:
class SGD:
    """
    SGD-衰减解耦
    """

    def __init__(self, learning_rate, weight_decay=0.0, momentum=0.0):
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.momentum = momentum
        self.momentum_W = None
        self.momentum_b = None

    def step(self, model):
        layers = model.layers
        if self.momentum_W is None or self.momentum_b is None:
            self.momentum_W, self.momentum_b = {}, {}
            for i, layer in enumerate(layers):
                if layer.trainable:
                    self.momentum_W[i] = np.zeros_like(layer.W)
                    self.momentum_b[i] = np.zeros_like(layer.b)

        for i, layer in enumerate(layers):
            if not layer.trainable:
                continue

            grad_W = layer.grad_W.copy()
            grad_b = layer.grad_b.copy()

            if self.weight_decay != 0:
                layer.W *= 1 - self.learning_rate * self.weight_decay

            if 0 < self.momentum <= 1:
                self.momentum_W[i] = grad_W + self.momentum * self.momentum_W[i]
                self.momentum_b[i] = grad_b + self.momentum * self.momentum_b[i]
                layer.W -= self.learning_rate * self.momentum_W[i]
                layer.b -= self.learning_rate * self.momentum_b[i]
            else:
                layer.W -= self.learning_rate * grad_W
                layer.b -= self.learning_rate * grad_b

## 2.2 `FCLayer`

`FCLayer` 为全连接层，输入为一组向量（必要时需要改变输入尺寸以满足要求），与权重矩阵作矩阵乘法并加上偏置项，得到输出向量:

$$
\mathbf{u} = \mathbf{W} \mathbf{x} + \mathbf{b}
$$


In [ ]:
class FCLayer:
    """
    全连接层
    """

    def __init__(
        self,
        num_input,
        num_output,
        act_function="relu",
        trainable=True,
        init_std=0.01,
        random_seed=2025,
    ):
        """
        对输入进行线性变换: y = Wx + b
        参数简介:
            num_input: 输入大小
            num_output: 输出大小
            act_function: 激活函数类型
            trainable: 是否具有可训练的参数
        """

        self.num_input = num_input
        self.num_output = num_output
        self.trainable = trainable
        self.act_function = act_function
        self.init_std = init_std
        self.random_seed = random_seed
        assert act_function in ["relu", "sigmoid"]

        self._xavier_init()

        self.grad_W = np.zeros((num_input, num_output))
        self.grad_b = np.zeros((1, num_output))

    def forward(self, input):
        ############################################################################
        # TODO
        # 对输入计算Wx+b并返回结果.
        self.input = input
        output = input @ self.W + self.b

        return output
        ############################################################################

    def backward(self, delta):
        # 输入的delta由下一层计算得到
        ############################################################################
        # TODO
        # 根据delta计算梯度
        self.grad_W = self.input.T @ delta
        self.grad_b = np.sum(delta, axis=0, keepdims=True)
        input_gradient = delta @ self.W.T

        return input_gradient
        ############################################################################

    def _xavier_init(self):
        # 初始化，无需了解.
        raw_std = (2 / (self.num_input + self.num_output)) ** 0.5
        if "relu" == self.act_function:
            self.init_std = raw_std * (2**0.5)
        elif "sigmoid" == self.act_function:
            self.init_std = raw_std
        else:
            self.init_std = raw_std

        np.random.seed(self.random_seed)
        self.W = np.random.normal(0, self.init_std, (self.num_input, self.num_output))
        self.b = np.random.normal(0, self.init_std, (1, self.num_output))

## 2.3 `SigmoidLayer`

`SigmoidLayer` 为 sigmoid 激活层:

$$
f(\mathbf{u}) = \frac{1}{1 + \exp(-\mathbf{u})}
$$


In [ ]:
class SigmoidLayer:
    """
    Sigmoid激活层
    """

    def __init__(self):
        """
        Sigmoid激活函数: f(x) = 1/(1+exp(-x))
        """

        self.trainable = False
        self.output = None

    def forward(self, input):
        ############################################################################
        # TODO
        # 对输入应用Sigmoid激活函数并返回结果
        positive_mask = input >= 0
        negative_mask = input < 0
        output = np.zeros_like(input)

        output[positive_mask] = 1.0 / (1.0 + np.exp(-input[positive_mask]))
        exp_input_neg = np.exp(input[negative_mask])
        output[negative_mask] = exp_input_neg / (1.0 + exp_input_neg)

        self.output = output

        return output
        ############################################################################

    def backward(self, delta):
        ############################################################################
        # TODO
        # 根据delta计算梯度

        return delta * self.output * (1 - self.output)
        ############################################################################

## 2.4 `ReLULayer`

`ReLULayer` 为 ReLU 激活层:

$$
f(\mathbf{u}) = \max(\mathbf{0}, \mathbf{u})
$$


In [ ]:
class ReLULayer:
    """
    ReLU激活层
    """

    def __init__(self):
        """
        ReLU激活函数: relu(x) = max(x, 0)
        """

        self.trainable = False

    def forward(self, input):
        ############################################################################
        # TODO
        # 对输入应用ReLU激活函数并返回结果
        self.mask = input > 0

        return np.maximum(0, input)
        ############################################################################

    def backward(self, delta):
        ############################################################################
        # TODO
        # 根据delta计算梯度

        return delta * self.mask
        ############################################################################

## 2.5 `EuclideanLossLayer`

`EuclideanLossLayer` 为欧式距离损失层，计算各样本误差的平方和得到:

$$
\frac{1}{2} \sum_{n} \lVert \operatorname{logits}(n) - \operatorname{label}(n) \rVert _2^2
$$


In [ ]:
class EuclideanLossLayer:
    """
    欧式距离损失层
    """

    def __init__(self):
        self.acc = 0.0
        self.loss = 0.0

    def forward(self, logit, gt):
        """
        输入: (minibatch)
        - logit: 最后一个全连接层的输出结果, 尺寸(batch_size, 10)
        - gt: 真实标签, 尺寸(batch_size, 10)
        """

        ############################################################################
        # TODO
        # 在minibatch内计算平均准确率和损失，分别保存在self.acc和self.loss里(将在训练时使用)
        # 只需要返回self.loss
        self.logit = logit
        self.gt = gt
        # 计算损失：平均欧式距离
        self.loss = np.mean(0.5 * np.sum((logit - gt) ** 2, axis=1))
        # 计算准确率：比较预测标签和真实标签
        pred_labels = np.argmax(logit, axis=1)
        true_labels = np.argmax(gt, axis=1)
        self.acc = np.mean(pred_labels == true_labels)
        return self.loss

        ############################################################################

    def backward(self):
        ############################################################################
        # TODO
        # 计算并返回梯度(与logit具有同样的尺寸)
        batch_size = self.logit.shape[0]

        return (self.logit - self.gt) / batch_size
        ############################################################################

## 2.6 `SoftmaxCrossEntropyLossLayer`

`SoftmaxCrossEntropyLossLayer` 可以看成是输入到如下概率分布的映射：

$$
P(t_k=1\vert\mathbf{x}) = \frac{\exp(x_k)}{\sum_{j=1}^{K}\exp(x_j)}
$$

其中 $x_k$ 是输入向量 $\mathbf{x}$ 中的第 $k$ 个元素，$P(t_k = 1|\mathbf{x})$ 表示该输入被分到第 $k$ 个类别的概率。由于 softmax 层的输出可以看成一组概率分布，我们可以计算 delta 似然及其对数形式，称为 Cross Entropy 误差函数：

$$
E = -\ln p(t^{(1)}, \dots, t^{(N)}) = \sum_{n=1}^N E^{(n)}
$$

其中

$$
E^{(n)} = -\sum_{k=1}^K t_k^{(n)} \ln h_k^{(n)}
$$

$$
h_k^{(n)} = P(t_k^{(n)}=1\vert \mathbf{x}^{(n)}) = \frac{\exp x_k^{(n)}}{\sum_{j=1}^K \exp x_j^{(n)}}
$$

注意：此处的 softmax 损失层与案例 1 中有所差异，本次案例中的 softmax 层不包含可训练的参数，这些可训练的参数被独立成一个全连接层。


In [ ]:
# 为了防止分母为零，必要时可在分母加上一个极小项EPS
EPS = 1e-8


class SoftmaxCrossEntropyLossLayer:
    """
    Softmax交叉熵损失层
    """

    def __init__(self):
        self.acc = 0.0
        self.loss = np.zeros(1, dtype="f")

    def forward(self, logit, gt):
        """
        输入: (minibatch)
        - logit: 最后一个全连接层的输出结果, 尺寸(batch_size, 10)
        - gt: 真实标签, 尺寸(batch_size, 10)
        """

        ############################################################################
        # TODO
        # 在minibatch内计算平均准确率和损失，分别保存在self.accu和self.loss里(将在训练时使用)
        # 只需要返回self.loss
        self.logit = logit
        self.gt = gt
        # 计算softmax，防止数值溢出：减去最大值
        shifted_logit = logit - np.max(logit, axis=1, keepdims=True)
        exp_logit = np.exp(shifted_logit)
        self.softmax = exp_logit / np.sum(exp_logit, axis=1, keepdims=True)
        # 计算损失：平均交叉熵
        self.loss = -np.mean(np.sum(gt * np.log(self.softmax + EPS), axis=1))
        # 计算准确率：比较预测标签和真实标签
        pred_labels = np.argmax(self.softmax, axis=1)
        true_labels = np.argmax(gt, axis=1)
        self.acc = np.mean(pred_labels == true_labels)
        return self.loss
        ############################################################################

    def backward(self):
        ############################################################################
        # TODO
        # 计算并返回梯度(与logit具有同样的尺寸)
        batch_size = self.logit.shape[0]

        return (self.softmax - self.gt) / batch_size
        ############################################################################


# 3. 训练代码


In [ ]:
def train_test(
    model, criterion, optimizer, max_epochs=30, batch_size=100, random_seed=2025
):
    all_train_losses, all_train_accs = [], []  # 记录所有批次的损失和准确率
    avg_train_losses, avg_train_accs = [], []  # 记录每个epoch的平均训练损失和准确率
    avg_val_losses, avg_val_accs = [], []  # 记录每个epoch的验证损失和准确率

    n_train = len(train_data)
    steps_per_epoch = np.ceil(n_train / batch_size).astype(int)

    # 训练循环
    np.random.seed(random_seed)
    for epoch in range(max_epochs):
        # 每个epoch开始时随机打乱训练数据
        np.random.seed(random_seed + epoch)
        indices = np.random.permutation(n_train)
        shuffled_train_data = train_data[indices]
        shuffled_train_labels = train_labels[indices]

        # 准备批次数据
        batch_split_indices = np.arange(batch_size, n_train, batch_size)
        train_data_batches = np.split(shuffled_train_data, batch_split_indices)
        train_labels_batches = np.split(shuffled_train_labels, batch_split_indices)

        batch_train_losses, batch_train_accs = [], []

        # 使用tqdm进度条
        progress_bar = tqdm.tqdm(
            range(steps_per_epoch), desc=f"Epoch[{epoch + 1}/{max_epochs}]"
        )
        for step_index in progress_bar:
            # 获取当前批次数据
            x = np.reshape(
                train_data_batches[step_index], (-1, 784)
            )  # 重塑为[batch_size, 784]
            y_true = np.eye(10)[
                train_labels_batches[step_index]
            ]  # 转换为one-hot编码，形状[batch_size, 10]

            # 前向传播
            y_pred = model.forward(x)
            criterion.forward(y_pred, y_true)

            # 反向传播
            delta = criterion.backward()
            model.backward(delta)

            # 优化器更新参数
            optimizer.step(model)

            # 记录当前批次的损失和准确率
            batch_train_losses.append(criterion.loss)
            batch_train_accs.append(criterion.acc)
            all_train_losses.append(criterion.loss)
            all_train_accs.append(criterion.acc)

            # 更新进度条描述
            progress_bar.set_description(
                f"Epoch[{epoch + 1}/{max_epochs}], "
                f"Train Loss: {criterion.loss:.3f}, "
                f"Train Acc: {criterion.acc:.2%}"
            )

        # 验证阶段：在每个epoch结束后评估验证集
        x_val = np.reshape(val_data, (-1, 784))
        y_true_val = np.eye(10)[val_labels]

        # 仅前向传播
        y_pred_val = model.forward(x_val)
        criterion.forward(y_pred_val, y_true_val)

        val_loss = criterion.loss
        val_acc = criterion.acc

        # 记录当前epoch的平均训练和验证指标
        avg_train_losses.append(np.mean(batch_train_losses))
        avg_train_accs.append(np.mean(batch_train_accs))
        avg_val_losses.append(val_loss)
        avg_val_accs.append(val_acc)

        print(f"  - Validation Loss: {val_loss:.3f}, Validation Acc: {val_acc:.2%}")

    # 测试阶段
    x_test = np.reshape(test_data, (-1, 784))
    y_true_test = np.eye(10)[test_labels]

    y_pred_test = model.forward(x_test)
    criterion.forward(y_pred_test, y_true_test)

    test_acc = criterion.acc
    print(f"\nTest Accuracy: {test_acc:.2%}\n")

    # 绘制损失和准确率曲线
    x_by_epoch = x_by_epoch = np.arange(max_epochs) + 1
    _, (ax0, ax1) = plt.subplots(1, 2, figsize=(18, 6), dpi=324)

    # 绘制损失曲线
    ax0.set_title("Loss Curves", fontsize=14, color="purple")
    ax0.plot(x_by_epoch, avg_train_losses, marker=".", label="Train", color="steelblue")
    ax0.plot(x_by_epoch, avg_val_losses, marker=".", label="Validation", color="orange")
    ax0.set_xticks(x_by_epoch)
    ax0.set_xlabel("Epochs", fontsize=12)
    ax0.grid(True, alpha=0.1)
    ax0.margins(x=0.02, y=0.1)
    ax0.legend()

    # 绘制准确率曲线
    ax1.set_title("Accuracy Curves", fontsize=14, color="purple")
    ax1.plot(x_by_epoch, avg_train_accs, marker=".", label="Train", color="steelblue")
    ax1.plot(x_by_epoch, avg_val_accs, marker=".", label="Validation", color="orange")
    ax1.set_xticks(x_by_epoch)
    ax1.set_xlabel("Epochs", fontsize=12)
    ax1.grid(True, alpha=0.1)
    ax1.margins(x=0.02, y=0.1)
    ax1.legend()

    plt.show()

    return {
        "model": model,  # 训练好的模型
        "train_losses": all_train_losses,  # 所有批次的训练损失列表
        "val_losses": avg_val_losses,  # 每个epoch的验证损失列表
        "train_accuracies": all_train_accs,  # 所有批次的训练准确率列表
        "val_accuracies": avg_val_accs,  # 每个epoch的验证准确率列表
        "steps_per_epoch": steps_per_epoch,  # 每个epoch的步数
    }

# 4. 实验与分析


## 1、实验分析一 -- 单隐层 MLP


### 1、不使用动量


#### 1、Sigmoid + EuclideanLoss


In [ ]:
# Sigmoid激活，欧式距离损失
# 128是隐藏层单元数，可根据需要修改
mlp_sigmoid_euclidean = Network()
mlp_sigmoid_euclidean.add(FCLayer(784, 128, act_function="sigmoid"))
mlp_sigmoid_euclidean.add(SigmoidLayer())
mlp_sigmoid_euclidean.add(FCLayer(128, 10, act_function="sigmoid"))

criterion_euclidean = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001)

(
    mlp_sigmoid_euclidean,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_sigmoid_euclidean, criterion_euclidean, sgd)

#### 2、Sigmoid + CrossEntropyLoss


In [ ]:
# Sigmoid激活，交叉熵损失
# 128是隐藏层单元数，可根据需要修改
mlp_sigmoid_ce = Network()
mlp_sigmoid_ce.add(FCLayer(784, 128))
mlp_sigmoid_ce.add(SigmoidLayer())
mlp_sigmoid_ce.add(FCLayer(128, 10))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001)

(
    mlp_sigmoid_ce,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_sigmoid_ce, criterion_ce, sgd)

#### 3、RELU + EuclideanLoss


In [ ]:
# ReLU激活，欧式距离损失
# 128是隐藏层单元数，可根据需要修改
mlp_relu_euclidean = Network()
mlp_relu_euclidean.add(FCLayer(784, 128, act_function="sigmoid"))
mlp_relu_euclidean.add(ReLULayer())
mlp_relu_euclidean.add(FCLayer(128, 10, act_function="sigmoid"))

criterion_euclidean = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001)

(
    mlp_relu_euclidean,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_relu_euclidean, criterion_euclidean, sgd)

#### 4、RELU + CrossEntropyLoss


In [ ]:
# ReLU激活，交叉熵损失
# 128是隐藏层单元数，可根据需要修改
mlp_relu_ce = Network()
mlp_relu_ce.add(FCLayer(784, 128))
mlp_relu_ce.add(ReLULayer())
mlp_relu_ce.add(FCLayer(128, 10))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001)

(
    mlp_relu_ce,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_relu_ce, criterion_ce, sgd)

### 2、使用动量


#### 1、Sigmoid + EuclideanLoss


In [ ]:
# Sigmoid激活，欧式距离损失
# 128是隐藏层单元数，可根据需要修改
mlp_sigmoid_euclidean = Network()
mlp_sigmoid_euclidean.add(FCLayer(784, 128, act_function="sigmoid"))
mlp_sigmoid_euclidean.add(SigmoidLayer())
mlp_sigmoid_euclidean.add(FCLayer(128, 10, act_function="sigmoid"))

criterion_euclidean = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp_sigmoid_euclidean,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_sigmoid_euclidean, criterion_euclidean, sgd)

#### 2、Sigmoid + CrossEntropyLoss


In [ ]:
# Sigmoid激活，交叉熵损失
# 128是隐藏层单元数，可根据需要修改
mlp_sigmoid_ce = Network()
mlp_sigmoid_ce.add(FCLayer(784, 128))
mlp_sigmoid_ce.add(SigmoidLayer())
mlp_sigmoid_ce.add(FCLayer(128, 10))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp_sigmoid_ce,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_sigmoid_ce, criterion_ce, sgd)

#### 3、RELU + EuclideanLoss


In [ ]:
# ReLU激活，欧式距离损失
# 128是隐藏层单元数，可根据需要修改
mlp_relu_euclidean = Network()
mlp_relu_euclidean.add(FCLayer(784, 128, act_function="sigmoid"))
mlp_relu_euclidean.add(ReLULayer())
mlp_relu_euclidean.add(FCLayer(128, 10, act_function="sigmoid"))

criterion_euclidean = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp_relu_euclidean,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_relu_euclidean, criterion_euclidean, sgd)

#### 4、RELU + CrossEntropyLoss


In [ ]:
# ReLU激活，交叉熵损失
# 128是隐藏层单元数，可根据需要修改
mlp_relu_ce = Network()
mlp_relu_ce.add(FCLayer(784, 128))
mlp_relu_ce.add(ReLULayer())
mlp_relu_ce.add(FCLayer(128, 10))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp_relu_ce,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp_relu_ce, criterion_ce, sgd)

### 3、总结对比


| **隐层激活函数** | **损失函数**     | **动量使用情况** | **测试准确率** |
| ---------------- | ---------------- | ---------------- | -------------- |
| Sigmoid          | EuclideanLoss    | 不使用动量       | 83.13%         |
| Sigmoid          | EuclideanLoss    | 使用动量         | 87.87%         |
| Sigmoid          | CrossEntropyLoss | 不使用动量       | 84.52%         |
| Sigmoid          | CrossEntropyLoss | 使用动量         | 91.47%         |
| RELU             | EuclideanLoss    | 不使用动量       | 89.24%         |
| RELU             | EuclideanLoss    | 使用动量         | 94.61%         |
| RELU             | CrossEntropyLoss | 不使用动量       | 90.52%         |
| RELU             | CrossEntropyLoss | 使用动量         | 95.48%         |


- **激活函数对比:** 对比可知，在相同损失函数和动量的情况下，使用 RELU 激活函数的单隐层 MLP 最终测试准确率要好于使用 Sigmoid 激活函数， 但不同激活函数对于 MLP 的收敛性影响不算显著；

- **损失函数对比:** 对比可知，在相同激活函数和动量的情况下，使用交叉熵损失函数的单隐层 MLP 最终测试准确率要好于使用欧氏距离损失函数；

- **动量使用对比:** 对比可知，在相同激活函数和损失函数的情况下，使用动量会加快模型的收敛速度，并且在训练 30 个 Epochs 后最终测试准确率好于不使用动量；

- **总结:** 使用 RELU 激活函数，交叉熵损失和动量组合的单隐层 MLP 在训练 30 个 Epochs 后最总测试准确率表现最好。


## 2、实验分析二 -- 双隐层 MLP


### 1、相同两层 -- 使用动量


#### 1、双 Sigmoid 隐层 + EuclideanLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(128, 64, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(64, 10, act_function="sigmoid"))

criterion_euclidean = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_euclidean, sgd)

#### 2、双 Sigmoid 隐层 + CrossEntropyLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(128, 64, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(64, 10, act_function="sigmoid"))

criterion_euclidean = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_euclidean, sgd)

#### 3、双 RELU 隐层 + EuclideanLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="sigmoid"))
mlp.add(ReLULayer())
mlp.add(FCLayer(128, 64, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(64, 10, act_function="relu"))

criterion_ce = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_ce, sgd)

#### 4、双 RELU 隐层 + CrossEntropyLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(128, 64, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(64, 10, act_function="relu"))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_ce, sgd)

### 2、不同两层 -- 使用动量


#### 1、Sigmoid + RELU + EuclideanLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(128, 64, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(64, 10, act_function="relu"))

criterion_ce = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_ce, sgd)

#### 2、Sigmoid + RELU + CrossEntropyLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(128, 64, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(64, 10, act_function="relu"))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_ce, sgd)

#### 3、RELU + Sigmoid + EuclideanLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(128, 64, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(64, 10, act_function="sigmoid"))

criterion_ce = EuclideanLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_ce, sgd)

#### 4、RELU + Sigmoid + CrossEntropyLoss


In [ ]:
mlp = Network()
mlp.add(FCLayer(784, 128, act_function="relu"))
mlp.add(ReLULayer())
mlp.add(FCLayer(128, 64, act_function="sigmoid"))
mlp.add(SigmoidLayer())
mlp.add(FCLayer(64, 10, act_function="sigmoid"))

criterion_ce = SoftmaxCrossEntropyLossLayer()
sgd = SGD(learning_rate=0.001, weight_decay=0.001, momentum=0.9)

(
    mlp,
    all_train_losses,
    avg_val_losses,
    all_train_accs,
    avg_val_accs,
    steps_per_epoch,
) = train_test(mlp, criterion_ce, sgd)

### 3、总结对比


| **第一隐层(使用动量)** | **第二隐层(使用动量)** | **损失函数**     | **测试准确率** |
| ---------------------- | ---------------------- | ---------------- | -------------- |
| Sigmoid                | Sigmoid                | EuclideanLoss    | 85.99%         |
| Sigmoid                | Sigmoid                | CrossEntropyLoss | 89.34%         |
| RELU                   | RELU                   | EuclideanLoss    | 94.51%         |
| RELU                   | RELU                   | CrossEntropyLoss | 96.47%         |
| Sigmoid                | RELU                   | EuclideanLoss    | 91.28%         |
| Sigmoid                | RELU                   | CrossEntropyLoss | 92.53%         |
| RELU                   | Sigmoid                | EuclideanLoss    | 92.74%         |
| RELU                   | Siomoid                | CrossEntropyLoss | 93.34%         |


**<font color="red">双隐层 MLP 表格的总结分析</font>**

- **激活函数对比：** 在损失函数和动量使用情况相同的前提下，采用 ReLU 激活函数组合（如 ReLU-ReLU）的测试准确率显著高于采用 Sigmoid 激活函数组合（如 Sigmoid-Sigmoid）。例如，在交叉熵损失函数下，ReLU-ReLU 组合的准确率为 96.47%，而 Sigmoid-Sigmoid 组合仅为 89.34%。混合激活函数组合（如 Sigmoid-ReLU 或 ReLU-Sigmoid）的表现介于两者之间，但仍优于纯 Sigmoid 组合，这表明 ReLU 激活函数在双隐层结构中具有明显优势。

- **损失函数对比：** 在激活函数组合和动量使用情况相同的前提下，使用交叉熵损失函数（CrossEntropyLoss）的测试准确率始终高于使用欧氏距离损失函数（EuclideanLoss）。例如，对于 ReLU-ReLU 组合，交叉熵损失函数对应的准确率为 96.47%，而欧氏距离损失函数仅为 94.51%。这一趋势在所有激活函数组合中均成立，凸显了交叉熵损失函数在分类任务中的有效性。

- **总结：** 在双隐层 MLP 中，使用 RELU-RELU 激活函数组合与交叉熵损失函数的配置表现最佳，测试准确率达到 96.47%。


**<font color="red">双隐层 MLP 与单隐层 MLP 的对比分析</font>**

&emsp;&emsp;将双隐层最佳配置（RELU-RELU + 交叉熵 + 动量，96.47%）与单隐层最佳配置（RELU + 交叉熵 + 动量，95.48%）进行对比，可以得出以下结论：

- **性能提升：** 增加一个隐藏层并均采用 RELU 激活函数，带来了模型性能的进一步提升，测试准确率提高了约 1%。这表明对于当前任务，更深的网络结构具有更强的表达能力。

- **最佳配置一致性：** 无论在单隐层还是双隐层网络中，RELU 激活函数配合交叉熵损失函数并加入动量，都是经过验证的最佳配置组合，这再次肯定了这些组件在训练神经网络中的有效性和重要性。

- **总结：** 综合来看，双隐层 MLP 在最优配置下取得了本次所有实验结果中的最佳性能，但其相对于性能优异的单隐层 MLP 的提升幅度有限。
